# 09 — Repeated Engine-Level Validation for RUL Target Cap

This notebook checks whether the current `cap_250`, `cap_350`, and `uncapped` ranking is stable across **three engine-level validation splits**. It keeps the Pseudo-Huber objective (`huber_slope=10`), row-uniform weighting, 10-cycle rolling window, regime-aware features, and XGBoost parameters fixed. Only target cap and split seed vary.

**Guardrails:**
- Uses FD001–FD004 training trajectories to obtain known validation RUL.
- Reads `test_FD00x.txt` only to sample observed terminal-cycle ages; does not load `RUL_FD00x.txt`.
- Reconstructs Notebook 07's locked audit-engine set and excludes those engines from every repetition.
- Fits regime clustering and sensor normalization on each split's development engines only.
- Does not overwrite or save a model.

This is a stability check, not a claim of statistical significance. Three splits are a practical preliminary comparison; larger repeated validation may be warranted if the finalists remain close.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from packaging.version import Version
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

if Version(xgb.__version__) < Version("1.7.0"):
    raise RuntimeError(f"xgboost>=1.7 is required for reg:pseudohubererror; found {xgb.__version__}.")

RANDOM_STATE = 20261026
SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
SENSOR_COLS = [f"s_{i}" for i in range(1, 22)]
SETTING_COLS = ["setting_1", "setting_2", "setting_3"]
WINDOW = 10
TARGET_CAP = 200  # overwritten per candidate in the experiment loop
N_REGIMES = 6
N_CUTOFFS_PER_ENGINE = 3

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CMAPSS_DIR = PROJECT_ROOT / "CMAPSSData"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in (RAW_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def locate_file(filename):
    for path in (RAW_DIR / filename, CMAPSS_DIR / filename, PROJECT_ROOT / filename):
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing {filename}; checked data/raw/, CMAPSSData/, and project root.")


def load_cmapss_file(filename):
    frame = pd.read_csv(locate_file(filename), sep=r"\s+", header=None, names=COLUMN_NAMES, engine="python")
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"Unexpected schema in {filename}: {frame.shape}")
    return frame


def load_training_data():
    parts = []
    for subset in SUBSETS:
        part = load_cmapss_file(f"train_{subset}.txt")
        part["subset"] = subset
        part["engine_key"] = subset + "_" + part["unit_nr"].astype(int).astype(str).str.zfill(3)
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


def load_test_terminal_ages_only():
    ages = {}
    rows = []
    for subset in SUBSETS:
        part = load_cmapss_file(f"test_{subset}.txt")
        terminal = part.groupby("unit_nr")["time_cycles"].max().astype(int).to_numpy()
        ages[subset] = terminal
        rows.append({"subset": subset, "test_engines": len(terminal), "median_observed_terminal_cycle": float(np.median(terminal))})
    return ages, pd.DataFrame(rows)

train_raw = load_training_data()
observed_test_ages, test_age_summary = load_test_terminal_ages_only()
print(f"XGBoost version: {xgb.__version__}")
print(f"Training rows: {len(train_raw):,} | training engines: {train_raw['engine_key'].nunique():,}")
display(test_age_summary)
print("Official test RUL label files were not loaded by this notebook.")

## 1. Build causal base features once

Rolling statistics are calculated in chronological order within each engine. At each validation cutoff, the model sees only features available by that cutoff. The lifetime-derived RUL column is a target/identifier column and is excluded from model inputs.

In [ ]:
def engineer_base_features(frame, window=WINDOW):
    ordered = frame.sort_values(["subset", "unit_nr", "time_cycles"]).reset_index(drop=True).copy()
    required = ["subset", "unit_nr", "engine_key", "time_cycles"] + SETTING_COLS + SENSOR_COLS
    features = ordered[required].copy()
    groups = ordered.groupby(["subset", "unit_nr"], sort=False)
    for sensor in SENSOR_COLS:
        signal = groups[sensor]
        features[f"{sensor}_roll_mean_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=1).mean())
        features[f"{sensor}_roll_std_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=2).std(ddof=0)).fillna(0.0)
        features[f"{sensor}_delta_1"] = signal.diff().fillna(0.0)
    for subset in SUBSETS:
        features[f"subset_{subset}"] = (ordered["subset"] == subset).astype(np.int8)
    final_cycle = ordered.groupby("engine_key")["time_cycles"].max()
    features["RUL_true"] = final_cycle.reindex(ordered["engine_key"]).to_numpy() - ordered["time_cycles"].to_numpy()
    return features


ID_COLS = {"subset", "unit_nr", "engine_key", "RUL_true"}
base = engineer_base_features(train_raw, WINDOW)
BASE_FEATURE_COLUMNS = [c for c in base.columns if c not in ID_COLS]
assert base[BASE_FEATURE_COLUMNS].notna().all().all()
assert not ({"engine_key", "unit_nr", "RUL_true", "subset"} & set(BASE_FEATURE_COLUMNS))

print(f"Base features prepared: {base.shape[0]:,} rows | {len(BASE_FEATURE_COLUMNS)} base features")
print("RUL labels are used only as training/validation targets; identifier and target columns are excluded from inputs.")


## 2. Build matched validation cutoffs and preserve the locked audit engines

For every repetition, development and validation engines are split **within each CMAPSS subset**. The audit engines assigned by Notebook 07's original split are excluded entirely from this experiment, so they are not re-used for model fitting or candidate selection.

In [ ]:
def build_age_matched_points(features, engine_keys, age_by_subset, n_per_engine=3, seed=420):
    rng = np.random.default_rng(seed)
    allowed = set(engine_keys)
    records = []
    use = features.loc[features["engine_key"].isin(allowed)]
    for engine_key, group in use.groupby("engine_key", sort=True):
        group = group.sort_values("time_cycles")
        subset = str(group["subset"].iloc[0])
        last_cycle = int(group["time_cycles"].max())
        empirical_ages = np.asarray(age_by_subset[subset], dtype=int)
        for replicate in range(n_per_engine):
            sampled_age = int(rng.choice(empirical_ages))
            cutoff = max(1, min(sampled_age, last_cycle - 1))
            candidates = group.loc[group["time_cycles"].le(cutoff)]
            if candidates.empty:
                continue
            row = candidates.iloc[-1]
            actual_cutoff = int(row["time_cycles"])
            records.append({"base_row_idx": int(row.name), "subset": subset, "engine_key": str(engine_key),
                            "unit_nr": int(row["unit_nr"]), "cutoff_cycle": actual_cutoff,
                            "sampled_test_age": sampled_age, "true_rul_cycles": float(last_cycle - actual_cutoff),
                            "replicate": int(replicate)})
    result = pd.DataFrame(records)
    if result.empty:
        raise ValueError("No pseudo-cutoff cases were generated.")
    return result


def make_locked_audit_split(seed=RANDOM_STATE):
    """Recreate Notebook 07's original locked audit engines. These engines are never used in this notebook."""
    locked_audit = []
    eligible = []
    for i, subset in enumerate(SUBSETS):
        keys = np.sort(train_raw.loc[train_raw["subset"].eq(subset), "engine_key"].unique())
        dev_keys, remainder = train_test_split(keys, test_size=0.40, random_state=seed + i)
        val_keys, audit_keys = train_test_split(remainder, test_size=0.50, random_state=seed + 100 + i)
        locked_audit.extend(audit_keys.tolist())
        eligible.extend(dev_keys.tolist())
        eligible.extend(val_keys.tolist())
    return set(eligible), set(locked_audit)

ELIGIBLE_KEYS, LOCKED_AUDIT_KEYS = make_locked_audit_split(RANDOM_STATE)
assert ELIGIBLE_KEYS.isdisjoint(LOCKED_AUDIT_KEYS)
print(f"Locked audit engines excluded from every repetition: {len(LOCKED_AUDIT_KEYS):,}")
print(f"Eligible engines for repeated development/validation splits: {len(ELIGIBLE_KEYS):,}")


## 3. Fit regime-aware features using development engines only

KMeans operating-regime assignments, setting scaling, and per-regime sensor normalization statistics are fitted only on the current split's development engines.

In [ ]:
def make_stats(part, source_columns):
    stats = {}
    for col in source_columns:
        vals = pd.to_numeric(part[col], errors="coerce").to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        stats[col] = (float(np.mean(vals)), max(float(np.std(vals, ddof=0)), 1e-6)) if len(vals) else (0.0, 1.0)
    return stats


def fit_regime_transformer(features, fit_engine_keys, random_state=RANDOM_STATE):
    source_columns = SENSOR_COLS + [f"{s}_roll_mean_{WINDOW}" for s in SENSOR_COLS]
    fit = features.loc[features["engine_key"].isin(set(fit_engine_keys))].copy().reset_index(drop=True)
    if fit.empty:
        raise ValueError("No data supplied to fit operating-regime transformer.")
    scaler = StandardScaler().fit(fit[SETTING_COLS])
    kmeans = KMeans(n_clusters=N_REGIMES, n_init=10, random_state=random_state)
    fit["_regime"] = kmeans.fit_predict(scaler.transform(fit[SETTING_COLS])).astype(int)
    group_stats = {}
    for (subset, regime), group in fit.groupby(["subset", "_regime"], sort=True):
        group_stats[(str(subset), int(regime))] = make_stats(group, source_columns)
    subset_stats = {str(s): make_stats(g, source_columns) for s, g in fit.groupby("subset", sort=True)}
    global_stats = make_stats(fit, source_columns)
    return {"setting_scaler": scaler, "kmeans": kmeans, "group_stats": group_stats,
            "subset_stats": subset_stats, "global_stats": global_stats, "n_regimes": N_REGIMES,
            "source_columns": source_columns, "rolling_window": WINDOW,
            "method": "KMeans operating regimes plus subset/regime sensor z-scores"}


def add_regime_features(features, transformer):
    out = features.copy().reset_index(drop=True)
    regime = transformer["kmeans"].predict(transformer["setting_scaler"].transform(out[SETTING_COLS])).astype(int)
    out["_regime"] = regime
    for r in range(int(transformer["n_regimes"])):
        out[f"operating_regime_{r}"] = (regime == r).astype(np.int8)
    positions = {}
    for pos, key in enumerate(zip(out["subset"].astype(str), regime)):
        positions.setdefault((key[0], int(key[1])), []).append(pos)
    for source_col in transformer["source_columns"]:
        means, stds = np.empty(len(out), dtype=float), np.empty(len(out), dtype=float)
        for key, locs in positions.items():
            stats = transformer["group_stats"].get(key, transformer["subset_stats"].get(key[0], transformer["global_stats"]))
            mean, sd = stats.get(source_col, transformer["global_stats"][source_col])
            means[locs], stds[locs] = mean, max(float(sd), 1e-6)
        out[f"{source_col}_regime_z"] = (out[source_col].to_numpy(dtype=float) - means) / stds
    out.drop(columns=["_regime"], inplace=True)
    return out


def feature_view(fit_engine_keys):
    transformer = fit_regime_transformer(base, fit_engine_keys)
    augmented = add_regime_features(base, transformer)
    regime_columns = [f"operating_regime_{i}" for i in range(N_REGIMES)] + [f"{c}_regime_z" for c in SENSOR_COLS + [f"{s}_roll_mean_{WINDOW}" for s in SENSOR_COLS]]
    columns = BASE_FEATURE_COLUMNS + regime_columns
    if augmented[columns].isna().any().any():
        raise ValueError("Non-finite features after regime transform.")
    return augmented, columns, transformer


## 4. Fixed metrics, score function, and XGBoost configuration

The NASA-style asymmetric score is reported beside MAE/RMSE and bias, never in isolation. The cap candidates are evaluated against **uncapped true RUL** on held-out training engines.

In [ ]:
def capped_target(y, cap=TARGET_CAP):
    y = np.asarray(y, dtype=float)
    return y if cap is None else np.minimum(y, float(cap))


def nasa_penalty_array(y_true, y_pred):
    error = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    exponent = np.where(error < 0, -error / 13.0, error / 10.0)
    return np.expm1(np.clip(exponent, 0.0, 50.0))


def score_points(points, predictions):
    scored = points[["subset", "engine_key", "unit_nr", "cutoff_cycle", "true_rul_cycles"]].copy().reset_index(drop=True)
    scored["predicted_rul_cycles"] = np.maximum(0.0, np.asarray(predictions, dtype=float))
    scored["error_pred_minus_true"] = scored["predicted_rul_cycles"] - scored["true_rul_cycles"]
    scored["absolute_error"] = scored["error_pred_minus_true"].abs()
    scored["squared_error"] = scored["error_pred_minus_true"] ** 2
    scored["nasa_penalty"] = nasa_penalty_array(scored["true_rul_cycles"], scored["predicted_rul_cycles"])
    scored["within_10_cycles"] = scored["absolute_error"] <= 10
    scored["within_20_cycles"] = scored["absolute_error"] <= 20
    scored["within_30_cycles"] = scored["absolute_error"] <= 30

    # First aggregate repeated cutoffs within each engine, then average engines equally within each subset.
    engine_metrics = scored.groupby(["subset", "engine_key"], as_index=False).agg(
        cases=("engine_key", "size"), mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        nasa_score_per_case=("nasa_penalty", "mean"),
        late_pct=("error_pred_minus_true", lambda x: float(np.mean(x > 0) * 100)),
        mean_bias=("error_pred_minus_true", "mean"),
        within_10_pct=("within_10_cycles", lambda x: float(np.mean(x) * 100)),
        within_20_pct=("within_20_cycles", lambda x: float(np.mean(x) * 100)),
        within_30_pct=("within_30_cycles", lambda x: float(np.mean(x) * 100)),
    )
    by_subset = engine_metrics.groupby("subset", as_index=False).agg(
        engines=("engine_key", "nunique"), cases=("cases", "sum"), mae=("mae", "mean"),
        rmse=("rmse", "mean"), nasa_score_per_case=("nasa_score_per_case", "mean"),
        late_pct=("late_pct", "mean"), mean_bias=("mean_bias", "mean"),
        within_10_pct=("within_10_pct", "mean"), within_20_pct=("within_20_pct", "mean"),
        within_30_pct=("within_30_pct", "mean"),
    )
    summary = {
        "n_cases": int(len(scored)), "n_engines": int(scored["engine_key"].nunique()),
        "pooled_mae": float(scored["absolute_error"].mean()),
        "pooled_rmse": float(np.sqrt(scored["squared_error"].mean())),
        "pooled_r2": float(r2_score(scored["true_rul_cycles"], scored["predicted_rul_cycles"])),
        "macro_subset_mae": float(by_subset["mae"].mean()),
        "macro_subset_rmse": float(by_subset["rmse"].mean()),
        "macro_subset_nasa_score_per_case": float(by_subset["nasa_score_per_case"].mean()),
        "macro_subset_late_pct": float(by_subset["late_pct"].mean()),
        "macro_subset_mean_bias": float(by_subset["mean_bias"].mean()),
        "macro_subset_within_10_pct": float(by_subset["within_10_pct"].mean()),
        "macro_subset_within_20_pct": float(by_subset["within_20_pct"].mean()),
        "macro_subset_within_30_pct": float(by_subset["within_30_pct"].mean()),
    }
    return summary, by_subset, scored, engine_metrics


def sample_weights(training_frame, policy):
    if policy == "row_uniform":
        return None
    if policy != "engine_balanced":
        raise ValueError(f"Unknown sample-weight policy: {policy}")
    per_engine_rows = training_frame.groupby("engine_key")["engine_key"].transform("size").to_numpy(dtype=float)
    weights = 1.0 / np.maximum(per_engine_rows, 1.0)
    return weights / weights.mean()

FIXED_XGB_PARAMS = {
    "n_estimators": 550,
    "max_depth": 5,
    "learning_rate": 0.03,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "min_child_weight": 5,
    "reg_lambda": 7.0,
    "tree_method": "hist",
    "n_jobs": 4,
}

LOSS_CANDIDATES = [
    {"loss_id": "squarederror", "objective": "reg:squarederror", "objective_params": {}},
    {"loss_id": "pseudohuber_slope10", "objective": "reg:pseudohubererror", "objective_params": {"huber_slope": 10.0}},
    {"loss_id": "absoluteerror", "objective": "reg:absoluteerror", "objective_params": {}},
    {"loss_id": "quantile_q035", "objective": "reg:quantileerror", "objective_params": {"quantile_alpha": 0.35}},
    {"loss_id": "quantile_q050", "objective": "reg:quantileerror", "objective_params": {"quantile_alpha": 0.50}},
]
WEIGHT_POLICIES = ["row_uniform", "engine_balanced"]

def weighted_quantile(values, quantile, weights=None):
    values = np.asarray(values, dtype=float)
    if weights is None:
        return float(np.quantile(values, quantile))
    weights = np.asarray(weights, dtype=float)
    order = np.argsort(values)
    sorted_values = values[order]
    sorted_weights = weights[order]
    cumulative = np.cumsum(sorted_weights)
    cutoff = float(quantile) * float(cumulative[-1])
    pos = min(int(np.searchsorted(cumulative, cutoff, side="left")), len(sorted_values) - 1)
    return float(sorted_values[pos])


def initial_base_score(candidate, y_fit, weights):
    """Set a loss-appropriate starting level; needed for stable robust-loss initialization."""
    objective = candidate["objective"]
    if objective == "reg:squarederror":
        return float(np.average(y_fit, weights=weights)) if weights is not None else float(np.mean(y_fit))
    if objective in {"reg:pseudohubererror", "reg:absoluteerror"}:
        return weighted_quantile(y_fit, 0.50, weights)
    if objective == "reg:quantileerror":
        q = float(candidate["objective_params"].get("quantile_alpha", 0.50))
        return weighted_quantile(y_fit, q, weights)
    raise ValueError(f"Unsupported objective in this experiment: {objective}")


def make_model(candidate, y_fit, weights=None, seed=RANDOM_STATE):
    params = dict(FIXED_XGB_PARAMS)
    params.update(candidate["objective_params"])
    params["objective"] = candidate["objective"]
    params["random_state"] = int(seed)
    # Explicit loss-appropriate intercept avoids unstable automatic initialization
    # for robust losses on RUL values measured in hundreds of cycles.
    params["base_score"] = initial_base_score(candidate, y_fit, weights)
    return XGBRegressor(**params)


def fit_candidate(candidate, fit_frame, feature_columns, fit_engine_keys, seed=RANDOM_STATE):
    mask = fit_frame["engine_key"].isin(set(fit_engine_keys))
    fit_rows = fit_frame.loc[mask]
    X_fit = fit_rows[feature_columns]
    y_fit = capped_target(fit_rows["RUL_true"].to_numpy(dtype=float), TARGET_CAP)
    w = sample_weights(fit_rows, candidate["weight_policy"])
    model = make_model(candidate, y_fit, weights=w, seed=seed)
    if w is None:
        model.fit(X_fit, y_fit)
    else:
        model.fit(X_fit, y_fit, sample_weight=w)
    return model

## 5. Run the repeated split comparison

The notebook trains 9 models in total (3 cap settings × 3 split seeds). This may take several minutes depending on the CPU. It records per-split metrics and a mean/standard-deviation summary; no model is saved.

In [ ]:
# Three repeated engine-level validation splits. Each split keeps the original Notebook 07 audit engines excluded.
# Only target cap varies among candidates; architecture, loss, sample weighting and XGBoost parameters remain fixed.
SPLIT_SEEDS = [20261026, 20261027, 20261028]
CAP_CANDIDATES = [250, 350, None]
FIXED_CANDIDATE = {
    "loss_id": "pseudohuber_slope10",
    "objective": "reg:pseudohubererror",
    "objective_params": {"huber_slope": 10.0},
    "weight_policy": "row_uniform",
}


def label_for_cap(cap):
    return "uncapped" if cap is None else f"cap_{int(cap)}"


def repeated_engine_split(seed):
    """Split only eligible engines; leave Notebook 07's locked audit engines completely untouched."""
    dev_keys, val_keys = [], []
    for i, subset in enumerate(SUBSETS):
        keys = np.asarray(sorted(k for k in ELIGIBLE_KEYS if k.startswith(subset + "_")), dtype=object)
        if len(keys) < 5:
            raise ValueError(f"Too few eligible engines for {subset}: {len(keys)}")
        dev, val = train_test_split(keys, test_size=0.25, random_state=int(seed) + i)
        dev_keys.extend(dev.tolist())
        val_keys.extend(val.tolist())
    assert set(dev_keys).isdisjoint(val_keys)
    assert set(dev_keys).isdisjoint(LOCKED_AUDIT_KEYS)
    assert set(val_keys).isdisjoint(LOCKED_AUDIT_KEYS)
    return dev_keys, val_keys


def build_summary_by_seed(summary, candidate_id, cap, split_seed):
    return {
        "split_seed": int(split_seed), "candidate_id": candidate_id, "target_cap": cap,
        "loss": FIXED_CANDIDATE["objective"], "huber_slope": 10.0,
        "weight_policy": FIXED_CANDIDATE["weight_policy"], "rolling_window": WINDOW,
        "feature_mode": "regime_aware", **summary,
    }

per_split_rows = []
prediction_frames = []
subset_frames = []

for split_idx, split_seed in enumerate(SPLIT_SEEDS, start=1):
    print(f"\\n===== Split {split_idx}/{len(SPLIT_SEEDS)} | seed={split_seed} =====")
    DEV_KEYS, VAL_KEYS = repeated_engine_split(split_seed)
    assert len(set(DEV_KEYS)) + len(set(VAL_KEYS)) <= len(ELIGIBLE_KEYS)

    # Fit regime clustering and normalization statistics on this split's development engines only.
    DEV_FEATURES, FEATURE_COLUMNS, DEV_TRANSFORMER = feature_view(DEV_KEYS)
    val_points = build_age_matched_points(
        base, VAL_KEYS, observed_test_ages,
        n_per_engine=N_CUTOFFS_PER_ENGINE, seed=int(split_seed) + 10,
    )
    X_validation = DEV_FEATURES.iloc[val_points["base_row_idx"].to_numpy(dtype=int)][FEATURE_COLUMNS]
    print(f"Development engines={len(set(DEV_KEYS)):,}; validation engines={len(set(VAL_KEYS)):,}; validation cases={len(val_points):,}")

    for cap_idx, cap in enumerate(CAP_CANDIDATES, start=1):
        TARGET_CAP = cap
        model_seed = int(split_seed) + 500
        model = fit_candidate(
            FIXED_CANDIDATE, DEV_FEATURES, FEATURE_COLUMNS, DEV_KEYS, seed=model_seed,
        )
        pred = np.maximum(0.0, model.predict(X_validation))
        summary, by_subset, scored, engine_metrics = score_points(val_points, pred)
        candidate_id = label_for_cap(cap)
        per_split_rows.append(build_summary_by_seed(summary, candidate_id, cap, split_seed))
        prediction_frames.append(scored.assign(split_seed=int(split_seed), candidate_id=candidate_id, target_cap=cap))
        subset_frames.append(by_subset.assign(split_seed=int(split_seed), candidate_id=candidate_id, target_cap=cap))
        print(
            f"  [{cap_idx}/{len(CAP_CANDIDATES)}] {candidate_id}: "
            f"macro MAE={summary['macro_subset_mae']:.3f}; "
            f"macro NASA={summary['macro_subset_nasa_score_per_case']:.4g}; "
            f"macro RMSE={summary['macro_subset_rmse']:.3f}"
        )

# Restore the project's original default after the comparison. No model is saved or overwritten.
TARGET_CAP = 200
per_split = pd.DataFrame(per_split_rows)
predictions_all = pd.concat(prediction_frames, ignore_index=True)
by_subset_all = pd.concat(subset_frames, ignore_index=True)

METRICS = [
    "pooled_mae", "pooled_rmse", "pooled_r2", "macro_subset_mae", "macro_subset_rmse",
    "macro_subset_nasa_score_per_case", "macro_subset_late_pct", "macro_subset_mean_bias",
    "macro_subset_within_20_pct",
]
summary_rows = []
for candidate_id, group in per_split.groupby("candidate_id", sort=False):
    row = {"candidate_id": candidate_id, "target_cap": group["target_cap"].iloc[0], "n_splits": int(group["split_seed"].nunique())}
    for metric in METRICS:
        row[f"{metric}_mean"] = float(group[metric].mean())
        row[f"{metric}_std"] = float(group[metric].std(ddof=1)) if len(group) > 1 else 0.0
        row[f"{metric}_min"] = float(group[metric].min())
        row[f"{metric}_max"] = float(group[metric].max())
    summary_rows.append(row)

stability_summary = pd.DataFrame(summary_rows).sort_values(
    ["macro_subset_mae_mean", "macro_subset_nasa_score_per_case_mean", "macro_subset_rmse_mean"],
    ascending=[True, True, True],
).reset_index(drop=True)

# Count split-wise wins to make stability visible; a single best average is not enough.
for metric in ["macro_subset_mae", "macro_subset_nasa_score_per_case", "macro_subset_rmse"]:
    best_idx = per_split.groupby("split_seed")[metric].idxmin()
    winners = per_split.loc[best_idx, "candidate_id"].value_counts()
    stability_summary[f"{metric}_split_wins"] = stability_summary["candidate_id"].map(winners).fillna(0).astype(int)

per_split_path = REPORTS_DIR / "target_cap_stability_per_split.csv"
summary_path = REPORTS_DIR / "target_cap_stability_summary.csv"
predictions_path = REPORTS_DIR / "target_cap_stability_predictions.csv"
subset_path = REPORTS_DIR / "target_cap_stability_by_subset.csv"
per_split.to_csv(per_split_path, index=False)
stability_summary.to_csv(summary_path, index=False)
predictions_all.to_csv(predictions_path, index=False)
by_subset_all.to_csv(subset_path, index=False)

display(per_split[["split_seed", "candidate_id", "pooled_mae", "pooled_rmse", "pooled_r2",
                   "macro_subset_mae", "macro_subset_rmse", "macro_subset_nasa_score_per_case",
                   "macro_subset_late_pct", "macro_subset_mean_bias"]].round(3))
print("\\n=== Repeated-split summary (mean/std across split seeds) ===")
display(stability_summary[["candidate_id", "n_splits", "macro_subset_mae_mean", "macro_subset_mae_std",
                           "macro_subset_rmse_mean", "macro_subset_rmse_std",
                           "macro_subset_nasa_score_per_case_mean", "macro_subset_nasa_score_per_case_std",
                           "macro_subset_late_pct_mean", "macro_subset_mean_bias_mean",
                           "macro_subset_mae_split_wins", "macro_subset_nasa_score_per_case_split_wins"]].round(3))
print("\\nReports saved:")
for path in [per_split_path, summary_path, predictions_path, subset_path]:
    print(" -", path)
print("No model artifact was saved or overwritten.")


## 6. Interpretation

- Compare `macro_subset_mae_mean` first if macro-subset MAE is your declared primary metric.
- Use `macro_subset_mae_std` and per-split wins to see whether the ranking is stable, not just the average.
- Compare the NASA score as a secondary safety-sensitive metric; a very large score can be dominated by a few high-RUL errors.
- If the candidates trade off MAE and NASA score, state that trade-off explicitly instead of claiming one configuration is universally best.
- This notebook does not select based on official test labels and does not update the deployed model.